In [73]:
# =============================================================================
# IMSS - Panel Histórico Completo
# Combina la lógica de IMSS_historico.py (varios meses) con la desagregación
# a sector 4 de IMSS_datos_abiertos_v1.py, todo en un solo DataFrame.
# =============================================================================

import pandas as pd
import gc

# =====================================================================
# 1. CONFIGURACIÓN: Archivos mensuales y diccionario
# =====================================================================
# Agrega o quita meses según necesites (formato "AAAA-MM-DD": ruta)
archivos_meses = {
    "2026-01-31": r"C:\Users\Alex\Documents\Programación\PAP\Actualización de datos\asg-2026-01-31.csv",
}

ruta_excel = "diccionario_de_datos_1.xlsx"

# =====================================================================
# 2. CARGA: Leemos y unimos todos los meses
# =====================================================================
lista_dataframes = []

for fecha, ruta in archivos_meses.items():
    print(f"Cargando datos de: {fecha}...")
    df_temp = pd.read_csv(ruta, sep='|', encoding='cp1252', low_memory=False)
    df_temp['fecha'] = fecha
    df_temp['cve_entidad'] = df_temp['cve_entidad'].astype(int) #Con datos antes de 2025 tienen un formato numérico, lo convertimos a string para evitar problemas al unir con datos posteriores.
    lista_dataframes.append(df_temp)

df_panel = pd.concat(lista_dataframes, ignore_index=True)
del lista_dataframes, df_temp
gc.collect()
print("¡Todos los meses han sido unidos!")



Cargando datos de: 2026-01-31...
¡Todos los meses han sido unidos!


In [75]:
# =====================================================================
# 3. DICCIONARIOS: Cargamos los catálogos desde Excel
# =====================================================================
# Entidades (con drop_duplicates para evitar duplicados por municipio)
df_muni = pd.read_excel(ruta_excel, sheet_name='entidad-municipio', skiprows=1)
df_muni['cve_entidad'] = df_muni['cve_entidad'].astype(int) #Aseguramos que la clave de entidad sea string para evitar problemas al unir con datos posteriores.
df_entidad = df_muni[['cve_entidad', 'descripción entidad']].drop_duplicates().reset_index(drop=True)

# Sector económico 1 (gran división)
df_sec1 = pd.read_excel(ruta_excel, sheet_name='sector 1', skiprows=1)

# Sector económico 4 (actividad detallada)
df_sec2 = pd.read_excel(ruta_excel, sheet_name='sector 2', skiprows=1)

del df_muni
gc.collect()
print("Diccionarios cargados.")

Diccionarios cargados.


In [76]:
# =====================================================================
# 4. AGRUPACIÓN: fecha + entidad + sector 1 + sector 2
# =====================================================================
columnas_agrupacion = ['fecha', 'cve_entidad', 'sector_economico_1', 'sector_economico_2']

df_resultado = (
    df_panel
    .groupby(columnas_agrupacion)[['asegurados', 'ta']]
    .sum()
    .reset_index()
)

# Liberamos el DataFrame grande que ya no necesitamos
del df_panel
gc.collect()
print("Agrupación completada.")

# =====================================================================
# 5. CRUCES: Enriquecemos con descripciones de los catálogos
# =====================================================================
# Cruce con Entidad
df_resultado = pd.merge(df_resultado, df_entidad, on='cve_entidad', how='left')

# Cruce con Sector 1
df_resultado = pd.merge(
    df_resultado,
    df_sec1[['sector_economico_1', 'descripción sector_economico_1']],
    on='sector_economico_1',
    how='left'
)

# Cruce con Sector 4 (llaves con nombres distintos)
df_resultado = pd.merge(
    df_resultado,
    df_sec2[['sector_economico_2_2pos', 'descripción sector_economico_2']],
    left_on='sector_economico_2',
    right_on='sector_economico_2_2pos',
    how='left'
)
df_resultado = df_resultado.drop(columns=['sector_economico_2_2pos'])

# Liberamos diccionarios
del df_entidad, df_sec1, df_sec2
gc.collect()
print("Cruces completados.")

# =====================================================================
# 6. ORDEN FINAL: Cronológico y por sector
# =====================================================================
df_resultado = df_resultado[[
    'fecha',
    'cve_entidad', 'descripción entidad',
    'sector_economico_1', 'descripción sector_economico_1',
    'sector_economico_2', 'descripción sector_economico_2',
    'asegurados', 'ta'
]]

df_resultado = df_resultado.sort_values(
    by=['sector_economico_1', 'sector_economico_2','fecha', 'cve_entidad']
).reset_index(drop=True)

print(f"\nPanel listo: {df_resultado.shape[0]:,} filas × {df_resultado.shape[1]} columnas")

Agrupación completada.
Cruces completados.

Panel listo: 1,909 filas × 9 columnas


In [77]:
# =====================================================================
# 7. (OPCIONAL) Filtrar por estado(s)
# =====================================================================
# Descomenta las siguientes líneas si solo necesitas datos de ciertos estados.
# Puedes agregar o quitar estados en la lista según necesites.

estados_filtro = ['Jalisco']
df_resultado = df_resultado[df_resultado['descripción entidad'].isin(estados_filtro)].reset_index(drop=True)
# print(f"Filtrado a {estados_filtro}: {df_resultado.shape[0]:,} filas restantes")

In [78]:
df_resultado.head()

,fecha,cve_entidad,descripción entidad,sector_economico_1,descripción sector_economico_1,sector_economico_2,descripción sector_economico_2,asegurados,ta
0,2026-01-31,14,Jalisco,0.0,"Div-Agricultura, Ganadería, Silvicultura, Pesc...",1.0,Agricultura,87500,87500
1,2026-01-31,14,Jalisco,0.0,"Div-Agricultura, Ganadería, Silvicultura, Pesc...",2.0,Ganadería,25622,25622
2,2026-01-31,14,Jalisco,0.0,"Div-Agricultura, Ganadería, Silvicultura, Pesc...",3.0,Silvicultura,386,386
3,2026-01-31,14,Jalisco,0.0,"Div-Agricultura, Ganadería, Silvicultura, Pesc...",4.0,Pesca,238,238
4,2026-01-31,14,Jalisco,0.0,"Div-Agricultura, Ganadería, Silvicultura, Pesc...",5.0,Caza,65,65


In [79]:
# =====================================================================
# 8. (OPCIONAL) Exportar a archivo
# =====================================================================
# df_resultado.to_excel('Panel_Historico_Completo.xlsx', index=False)
df_resultado.to_csv('2026_Enero.csv', index=False, encoding='latin1')